# 🎵 Laxman Lofi AI Studio — GPU Runner

This runner provides **AI lyrics + ACE-Step music generation + server-side mastering/export** from one API.

The runner now detects GPU capability automatically. NVIDIA T4 uses FP32/FP16-compatible settings instead of forcing BF16, and startup errors are written to `/content/laxman-lofi-api.log`.

In [ ]:
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq ffmpeg
!git clone https://github.com/LaxmanNepal/lofi.git /content/lofi || (cd /content/lofi && git pull --ff-only)
!git clone https://github.com/ace-step/ACE-Step.git /content/ACE-Step || (cd /content/ACE-Step && git pull --ff-only)
%cd /content/lofi
!pip -q install -e /content/ACE-Step
!pip -q install -r backend/requirements.txt nest-asyncio


In [ ]:
import os, sys, subprocess, time, requests
import torch
os.environ['ACE_CHECKPOINT_DIR']='/content/ace-checkpoints'
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    os.environ['ACE_BF16']='true' if major >= 8 else 'false'
    print(f'GPU: {torch.cuda.get_device_name(0)} | compute {major}.{minor} | ACE_BF16={os.environ["ACE_BF16"]}')
else:
    os.environ['ACE_BF16']='false'
    print('WARNING: CUDA GPU not detected')
os.environ['LYRIC_MODEL_ID']='ministral/Ministral-3b-instruct'
os.environ['LYRIC_4BIT']='true'
# Verify the same import used by backend.server before starting Uvicorn.
from acestep.pipeline_ace_step import ACEStepPipeline
print('ACE-Step import: OK')
# Optional API protection. Never commit the secret.
# os.environ['LAXMAN_LOFI_API_KEY']='change-this-secret'
api_log=open('/content/laxman-lofi-api.log','w')
api=subprocess.Popen([sys.executable,'-m','uvicorn','backend.server:app','--host','0.0.0.0','--port','8000'],stdout=api_log,stderr=subprocess.STDOUT)
time.sleep(8)
if api.poll() is not None:
    api_log.flush()
    print(open('/content/laxman-lofi-api.log').read())
    raise RuntimeError('Laxman Lofi API failed to start')
print(requests.get('http://127.0.0.1:8000/health',timeout=10).text)


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
import subprocess, time
log=open('/content/cloudflared.log','w')
subprocess.Popen(['cloudflared','tunnel','--url','http://127.0.0.1:8000'],stdout=log,stderr=subprocess.STDOUT)
time.sleep(6)
print(open('/content/cloudflared.log').read())


## Connect the web app

Copy the `https://....trycloudflare.com` URL from the previous cell. Open **Laxman Lofi → Settings (⚙)** and paste it into **ACE-Step API URL**.

Keep the Colab runtime alive while generating. If generation fails, inspect `/content/laxman-lofi-api.log` for the exact backend traceback.